### Data-Level Security Using Dynamic Views

This notebook demonstrates how data-level security can be implemented using dynamic vies.

The implementation coers two key capabilities:
- Row-Level Security: Constrols which rows are visible to a user
- Column-level masking: Conttrols how sensitive values are displayed


A simple sales datase is used to illustrate how different users receie diferent results when querying the same view

#### Step 1: Set up the schema for the table

In [0]:
USE CATALOG demo;
CREATE SCHEMA IF NOT EXISTS data_security;

In [0]:
USE SCHEMA data_security;

#### Step 2: Create the table

A table is defined to represent sales data across multiple regions.

In [0]:
CREATE OR REPLACE TABLE sales_dynamic (
    id INT,
    region STRING,
    email STRING,
    revenue INT
)

#### Step 3: Insert sample data

The dataset includes records from both US and US regions.

This enables validation of row-level security by ensuring that different users see only relevant regional data.

In [0]:
INSERT INTO sales_dynamic VALUES
    (1, 'UK', 'jane@sales.com', 100),
    (2, 'US', 'bob@sales.com', 200),
    (3, 'US', 'jane@sales.com', 300),
    (4, 'UK', 'bob@sales.com', 400),
    (5, 'US', 'jane@sales.com', 500),
    (6, 'UK', 'bob@sales.com', 600),
    (7, 'US', 'jane@sales.com', 700),
    (8, 'UK', 'bob@sales.com', 800),
    (9, 'US', 'jane@sales.com', 900),
    (10, 'UK', 'bob@sales.com', 1000)

In [0]:
SELECT * FROM sales_dynamic;


### Step 4: Create the dynamic view

The dynamic view implements both row-level security and column-level masking

###### Row-Level Securty 

- UK group -> returns only rows where region = 'UK'
- US group -> returns only rows where region = 'US'
- Admin group -> returns all rows


###### Column-level Masking

- Admin group -> full email visible
- Other users -> email is masked

###### Key Function: is_account_group_member()

This function evaluates wheter the current user belongs to a specific group.

Examples:

- is_account_group_member('uk-sg')
- is_account_group_member('us-sg')
- is_account_group_member('admin-sg')

The function returls TRUE or FALSE and is used to drive conditional logic inside the view

In [0]:
SELECT is_account_group_member('admin-sg')

In [0]:
CREATE OR REPLACE VIEW vw_sales_dynamic AS
SELECT  id,
        region,
        CASE 
            WHEN is_account_group_member('admin-sg') THEN email
            ELSE concat(substr(email, 1, 1), '***@', split(email, '@')[1])
        END AS email,
        revenue
FROM sales_dynamic
WHERE is_account_group_member('admin-sg')
OR (is_account_group_member('uk-sg') AND region = 'UK')
OR (is_account_group_member('us-sg') AND region = 'US');


##### Step 5: Query the dynamic view

In [0]:
SELECT * FROM vw_sales_dynamic;